# Agent 3 — Create Databricks Workflow

Creates the on-demand SQL review workflow.
Agent 3 is triggered manually by a developer
when they want to submit a transformation for review.

Workflow structure:
  Task 1: run_sql_review_agent
  Task 2: run_alert_dispatcher (depends on Task 1)

In [0]:
import requests
import json

host  = spark.conf.get("spark.databricks.workspaceUrl")
token = (
    dbutils.notebook.entry_point
           .getDbutils().notebook().getContext()
           .apiToken().get()
)

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type" : "application/json"
}

EXISTING_CLUSTER_ID = "0220-160255-4reafcow"

ROOT      = "/Users/s819534@8451.com/data_sentinel_barrows"
PIPE_PATH = f"{ROOT}/agent3/pipeline"

SQL_REVIEW_PATH  = f"{PIPE_PATH}/01_sql_review_agent"
ALERT_DISP_PATH  = f"{PIPE_PATH}/02_alert_dispatcher"

NOTIFICATION_EMAIL = "s819534@8451.com"

print("=" * 60)
print("AGENT 3 — WORKFLOW CREATION")
print("=" * 60)
print(f"Cluster ID       : {EXISTING_CLUSTER_ID}")
print(f"SQL review path  : {SQL_REVIEW_PATH}")
print(f"Alert disp path  : {ALERT_DISP_PATH}")
print("=" * 60)

## Step 1: Verify Notebooks Exist

In [0]:
def verify_notebook(path: str) -> bool:
    response = requests.get(
        f"https://{host}/api/2.0/workspace/get-status",
        headers=headers,
        params={"path": path}
    )
    exists = response.status_code == 200
    status = "EXISTS  ✅" if exists else "MISSING ❌"
    print(f"  {status}  {path}")
    return exists

print("Verifying notebooks...")
all_exist = all([
    verify_notebook(SQL_REVIEW_PATH),
    verify_notebook(ALERT_DISP_PATH)
])

if not all_exist:
    raise Exception(
        "One or more notebooks missing. "
        "Create them before creating the workflow."
    )
print("\nAll notebooks verified.")

## Step 2: Create Agent 3 Workflow

In [0]:
workflow = {
    "name": "agent3_silver_sql_review",

    # ── No schedule — on demand only ──────────────────────────────────────
    # Triggered manually by developer via:
    #   1. Databricks Jobs UI (Run Now button)
    #   2. REST API with transformation_id parameter
    #   3. Admin UI trigger button (future enhancement)

    "email_notifications": {
        "on_failure"               : [NOTIFICATION_EMAIL],
        "on_success"               : [],
        "no_alert_for_skipped_runs": True
    },

    "max_concurrent_runs": 3,       # Allow parallel reviews
    "timeout_seconds"    : 3600,

    "tags": {
        "team"   : "data-engineering",
        "project": "data-trust-layer",
        "agent"  : "agent3"
    },

    # ── Job Parameters ─────────────────────────────────────────────────────
    # Single source of truth for target catalog/schema. Exposed to every
    # notebook task below as widgets ("catalog" / "schema"), so promoting
    # the whole job to a new catalog/schema only requires editing these
    # two defaults instead of every notebook.
    "parameters": [
        {"name": "catalog", "default": "learning_dev"},
        {"name": "schema"  , "default": "s819534"}
    ],

    "tasks": [

        # ── Task 1: SQL Review Agent ───────────────────────────────────────
        {
            "task_key"           : "run_sql_review_agent",
            "description"        : (
                "Reviews Silver transformation SQL for risky patterns "
                "and validates row count survival rates."
            ),
            "existing_cluster_id": EXISTING_CLUSTER_ID,
            "notebook_task"      : {
                "notebook_path"   : SQL_REVIEW_PATH,
                "base_parameters" : {
                    "transformation_id": "",   # blank = all active
                    "business_date"    : ""    # blank = yesterday
                },
                "source"          : "WORKSPACE"
            },
            "timeout_seconds"          : 2400,
            "max_retries"              : 1,
            "min_retry_interval_millis": 120000,
            "retry_on_timeout"         : False,
            "email_notifications"      : {
                "on_failure": [NOTIFICATION_EMAIL]
            }
        },

        # ── Task 2: Alert Dispatcher ───────────────────────────────────────
        {
            "task_key"           : "run_alert_dispatcher",
            "description"        : (
                "Sends Teams alerts for CRITICAL and WARNING "
                "SQL review results."
            ),
            "existing_cluster_id": EXISTING_CLUSTER_ID,
            "depends_on"         : [
                {"task_key": "run_sql_review_agent"}
            ],
            "notebook_task"      : {
                "notebook_path": ALERT_DISP_PATH,
                "source"       : "WORKSPACE"
            },
            "timeout_seconds"          : 900,
            "max_retries"              : 2,
            "min_retry_interval_millis": 60000,
            "retry_on_timeout"         : False
        }
    ]
}

print("Creating Agent 3 workflow...")
response = requests.post(
    f"https://{host}/api/2.1/jobs/create",
    headers=headers,
    json=workflow
)

print(f"Status code : {response.status_code}")

if response.status_code == 200:
    job_id = response.json().get("job_id")
    print(f"\n✅ Workflow created.")
    print(f"   Job ID  : {job_id}")
    print(f"   View at : https://{host}/#job/{job_id}")
    print()
    print("No schedule set — triggered on demand.")
    print("To trigger: pass transformation_id as a parameter.")
else:
    print(f"\n❌ Failed: {response.text}")
    job_id = None

## Step 3: Test Manual Trigger

In [0]:
if job_id:
    trigger_payload = {
        "job_id"          : int(job_id),
        "notebook_params" : {
            "transformation_id": "",          # all transformations
            "business_date"    : "2026-06-21" # a date with data
        }
    }

    print(f"Triggering test run...")
    trigger_resp = requests.post(
        f"https://{host}/api/2.1/jobs/run-now",
        headers=headers,
        json=trigger_payload
    )

    print(f"Status code : {trigger_resp.status_code}")

    if trigger_resp.status_code == 200:
        run_id = trigger_resp.json().get("run_id")
        print(f"\n✅ Test run triggered.")
        print(f"   Run ID  : {run_id}")
        print(f"   View at : https://{host}/#job/{job_id}/run/{run_id}")
    else:
        print(f"\n❌ Trigger failed: {trigger_resp.text}")
        run_id = None

## Step 4: Monitor the Run

In [0]:
import time
from datetime import datetime

def monitor_run(run_id: int, poll_seconds: int = 15):
    if not run_id:
        print("No run_id. Trigger a run first.")
        return

    terminal_states = {"TERMINATED", "SKIPPED", "INTERNAL_ERROR"}

    task_emojis = {
        "run_sql_review_agent" : "🧪",
        "run_alert_dispatcher" : "📣"
    }

    status_emojis = {
        "RUNNING"       : "⏳",
        "TERMINATED"    : "✅",
        "INTERNAL_ERROR": "❌",
        "PENDING"       : "⏸"
    }

    print(f"Monitoring run {run_id}...")
    previous_states = {}

    while True:
        resp = requests.get(
            f"https://{host}/api/2.1/jobs/runs/get",
            headers=headers,
            params={"run_id": run_id}
        )

        if resp.status_code != 200:
            print(f"Polling error: {resp.status_code}")
            break

        run_data   = resp.json()
        state      = run_data.get("state", {})
        life_state = state.get("life_cycle_state", "UNKNOWN")
        result     = state.get("result_state", "")
        tasks      = run_data.get("tasks", [])

        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}] Overall: {life_state} {result}")

        for task in tasks:
            task_key   = task.get("task_key", "")
            task_state = task.get("state", {})
            task_life  = task_state.get("life_cycle_state", "PENDING")
            task_result= task_state.get("result_state", "")
            t_emoji    = task_emojis.get(task_key, "⚙️")
            s_emoji    = status_emojis.get(task_life, "❓")

            current = f"{task_life} {task_result}"
            if previous_states.get(task_key) != current:
                previous_states[task_key] = current
                print(
                    f"  {t_emoji} {task_key:<30} "
                    f"{s_emoji} {task_life:<15} "
                    f"{task_result}"
                )

        print()

        if life_state in terminal_states:
            print("=" * 50)
            if result == "SUCCESS":
                print("✅ Agent 3 workflow completed successfully.")
            else:
                print(f"❌ Workflow ended: {result}")
                msg = state.get("state_message", "")
                if msg:
                    print(f"   Message: {msg}")
            print("=" * 50)
            break

        time.sleep(poll_seconds)


if run_id:
    monitor_run(int(run_id))